# Multiple backgrounds with a concise workflow

`ToyBackground` and `BackgroundSpec` use the same remainder convention, so generation and fitting of several backgrounds no longer require manual normalization arrays.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BackgroundSpec, DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    FitSession, ToyBackground, enable_x64, generate_toy,
    plot_dalitz, plot_square_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground

enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [Resonance("Kstar",(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),
     NonResonant(RealImag(-0.25,0.10))],
    normalization_method="square-dalitz",normalization_resolution=180,normalization_pair=(0,2),
)
comb=FunctionalBackground(lambda d:0.3+0.7*jnp.clip(d["s23"]/25,0,1))
partial=FunctionalBackground(lambda d:jnp.exp(-0.12*jnp.clip(d["s13"],0,None)))
misid=FunctionalBackground(lambda d:0.5+0.5*jnp.cos(0.15*d["s13"])**2)


In [ ]:
toy=generate_toy(
    model,30_000,signal_fraction=0.75,
    backgrounds=(
        ToyBackground("comb",comb,fraction=0.50),
        ToyBackground("partial",partial,fraction=0.30),
        ToyBackground("misID",misid),
    ),
    seed=808,pool_size=220_000,
)
plot_dalitz(toy,x="s13",y="s23",title="Three-background toy")
plt.show()


In [ ]:
f_sig=Parameter("signal_fraction",0.68,bounds=(0.05,0.98))
f_comb=Parameter("comb_fraction",0.45,bounds=(0.20,0.70))
f_partial=Parameter("partial_fraction",0.25,bounds=(0.10,0.30))
session=FitSession(
    model,toy,signal_fraction=f_sig,
    backgrounds=(
        BackgroundSpec("comb",comb,fraction=f_comb),
        BackgroundSpec("partial",partial,fraction=f_partial),
        BackgroundSpec("misID",misid),
    ),
)
result=session.fit(simplex=True,ncall=30_000)
session.report(result)
session.plot_projection(result,"s13")
plt.show()
